# Final Results: TF-IDF vs Dual-Encoder

Loads the unified comparison table from `docs/reports/evaluation_results.csv`.

Regenerate with: `python src/evaluate.py`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('../..').resolve()
RESULTS_PATH = PROJECT_ROOT / 'docs/reports/evaluation_results.csv'

results = pd.read_csv(RESULTS_PATH)
display_cols = ['Model', 'Query type', 'Top-1', 'Top-5', 'MRR', 'Precision@5']
results[display_cols].round(3)

In [ ]:
metrics = ['Top-1', 'Top-5', 'MRR', 'Precision@5']
query_types = results['Query type'].unique()
x = range(len(query_types))
width = 0.35

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    baseline = results[results['Model'] == 'TF-IDF'].set_index('Query type')[metric]
    dual = results[results['Model'].str.contains('Dual')].set_index('Query type')[metric]
    ax.bar([i - width/2 for i in x], [baseline.get(q, 0) for q in query_types], width, label='TF-IDF')
    ax.bar([i + width/2 for i in x], [dual.get(q, 0) for q in query_types], width, label='Dual-encoder')
    ax.set_xticks(list(x))
    ax.set_xticklabels(query_types, rotation=15)
    ax.set_title(metric)
    ax.set_ylim(0, 1)
    ax.legend()

plt.suptitle('Retrieval performance on test set (4,427 products)', fontsize=14)
plt.tight_layout()
plt.show()

## Experiment progression (v1 → v5)

During development we iterated through five model versions (notebooks in `richardson_experiment/`). Below: validation-set Recall@K from the ablation runs. The final unified test evaluation is in the table above.

See `notebooks/richardson_experiment/README.md` for the full notebook-by-notebook story.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ABLATIONS_DIR = PROJECT_ROOT / 'docs/reports/ablations'

v134 = pd.read_csv(ABLATIONS_DIR / 'v1_v2_v3_v4_comparison.csv')
final = pd.read_csv(ABLATIONS_DIR / 'final_comparison.csv')

display(v134.round(3))
display(final.round(3))

# v1 → v4 progression (template query)
template_rows = v134[v134['Setup'].str.contains('template query', case=False)]
labels = ['v1', 'v2', 'v3', 'v4']
r1 = template_rows['R@1'].values

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, r1, color=['#94a3b8', '#64748b', '#475569', '#1e293b'])
ax.set_ylabel('Recall@1 (validation)')
ax.set_title('Ablation progression: templated query style')
ax.set_ylim(0, max(r1) * 1.2)
for i, v in enumerate(r1):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Retrieval demo (dual-encoder)

Requires trained weights at `models/v4_image_encoder.weights.h5`.

In [ ]:
import sys
import numpy as np
from PIL import Image

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from captions import caption_shopper
from search import load_dual_encoder
from model import encode_images, encode_texts, resolve_image_path

test_df = pd.read_csv(PROJECT_ROOT / 'data/processed/test.csv')
query_row = test_df.sample(1, random_state=42).iloc[0]
query = caption_shopper(query_row)
print('Query:', query)
print('Target product:', query_row['productDisplayName'])

In [ ]:
weights = PROJECT_ROOT / 'models/v4_image_encoder.weights.h5'
if not weights.exists():
    print('Train first: python src/train.py')
else:
    model, text_encoder = load_dual_encoder()
    image_paths = [resolve_image_path(p, PROJECT_ROOT) for p in test_df['image_path']]
    image_emb = np.load(PROJECT_ROOT / 'models/test_image_embeddings.npy')
    if image_emb.shape[0] != len(test_df):
        image_emb = encode_images(model, image_paths)

    q_emb = encode_texts(text_encoder, [query])
    sims = (q_emb @ image_emb.T).flatten()
    top_idx = sims.argsort()[::-1][:5]

    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for ax, idx in zip(axes, top_idx):
        row = test_df.iloc[idx]
        img = Image.open(resolve_image_path(row['image_path'], PROJECT_ROOT))
        ax.imshow(img)
        ax.axis('off')
        title = str(row['productDisplayName'])[:30]
        if row['id'] == query_row['id']:
            title += '\n(correct)'
        ax.set_title(title, fontsize=8)
    plt.suptitle(f'Query: {query}')
    plt.tight_layout()
    plt.show()